### We load and encode our data

In [1]:
with (
    open("./data_train.txt", "r", encoding="utf-8") as f_train,
    open("./data_val.txt", "r", encoding="utf-8") as f_val,
):
    text_train = f_train.read()
    text_val = f_val.read()

print("Num of chars in dataset: ", len(text_train))
print("Num of chars in validation: ", len(text_val))

chars = sorted(list(set(text_train)))
vocab_size = len(chars)
print("Vocab: ", chars)
print("Vocab size: ", vocab_size)

Num of chars in dataset:  5429319
Num of chars in validation:  23291
Vocab:  ['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}']
Vocab size:  94


In [2]:
# Encode and decode data
stoi = {c:i for i, c in enumerate(chars)}
itos = {i:c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("Hello world !"))
print(decode(encode("Hello world !")))


[40, 69, 76, 76, 79, 1, 87, 79, 82, 76, 68, 1, 2]
Hello world !


### We create a function to return train and validation batches

In [3]:
import torch 

train_data = torch.tensor(encode(text_train), dtype=torch.long)
val_data = torch.tensor(encode(text_val), dtype=torch.long)
print(train_data.shape, train_data.dtype)


block_size = 8
x= train_data[:block_size]
y= train_data[1:block_size+1]

print(x, y)

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When the input is {context} the target is {target}")

torch.Size([5429319]) torch.int64
tensor([47, 76, 68, 69, 82,  1, 51, 73]) tensor([76, 68, 69, 82,  1, 51, 73, 66])
When the input is tensor([47]) the target is 76
When the input is tensor([47, 76]) the target is 68
When the input is tensor([47, 76, 68]) the target is 69
When the input is tensor([47, 76, 68, 69]) the target is 82
When the input is tensor([47, 76, 68, 69, 82]) the target is 1
When the input is tensor([47, 76, 68, 69, 82,  1]) the target is 51
When the input is tensor([47, 76, 68, 69, 82,  1, 51]) the target is 73
When the input is tensor([47, 76, 68, 69, 82,  1, 51, 73]) the target is 66


In [4]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

xb, yb = get_batch("train")
print("INPUTS: ")
print(xb.shape)
print(xb)
print("TARGETS: ")
print(yb.shape)
print(yb)

INPUTS: 
torch.Size([4, 8])
tensor([[73, 85, 68, 27,  1, 55, 79, 87],
        [73, 71, 72, 84,  1, 66, 65, 78],
        [69, 89,  1, 87, 69, 78, 84,  1],
        [ 1, 76, 79, 86, 69, 68,  1, 72]])
TARGETS: 
torch.Size([4, 8])
tensor([[85, 68, 27,  1, 55, 79, 87, 13],
        [71, 72, 84,  1, 66, 65, 78, 71],
        [89,  1, 87, 69, 78, 84,  1, 72],
        [76, 79, 86, 69, 68,  1, 72, 69]])


### A simple bigram model

In [5]:

import torch 
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_chars):
        for _ in range(max_chars):
            out, _ = self(idx)
            probs = torch.softmax(out[:, -1, :], dim=1)
            new_idx = torch.multinomial(probs, num_samples=1)   
            idx = torch.concatenate((idx, new_idx), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(loss)

print(decode(map(int, m.generate(torch.tensor([encode("L")]), 100)[0])))

torch.Size([32, 94])
tensor(5.0583, grad_fn=<NllLossBackward0>)
L$#Mcvy!ZeL}n_Rw8/ wTnV/S*-v8]`'trt$#s^!gmLW:jm_7Vu6X9}J+cL%RY|0)!`'M?i,i>
PdfO/\otd^iU:rSB}%R.VW0hLP


In [6]:
epochs = 10_000

optimizer = torch.optim.AdamW(m.parameters(), lr=0.001)

for epoch in range(epochs):
    X, Y = get_batch("train")

    out, loss = m(X, Y)


    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    optimizer.step()

    print("Loss: %.8f" % (loss.item())) if epoch % 1000 == 0 else ...



Loss: 4.94117069
Loss: 4.56336975
Loss: 3.99491501
Loss: 3.27430320
Loss: 2.70485234
Loss: 2.76384330
Loss: 2.76438165
Loss: 2.69012213
Loss: 2.78600311
Loss: 2.42165637


In [7]:
print(decode(map(int, m.generate(torch.tensor([encode("L")]), 500)[0])))

L s tombom the itcam]j5Basand "y Acomer s ig akes okARlabung hy.
BMaobwhicigMEmongheloto MAtiolwarerene m il yeowa t tarmp 3:0G2Pve atis?
GAs ay WK9^(, beke m:Limurithe herewane t issp. ithed ware be
N
>Fly gat a d ndatifY&)etour waslurnd, qhe?
Emsshedirourid awiatresto tlL]WNe trt! OU'q, hay, kesm'sor crtulloQs e catuecket iring htey.
VDar. ang nt*Dd she! yore catathand?
Zgoly bwhayShldoow Shond 's wh}ly suped ck!
ALakamme o "]4\?jz32Cmend!

4>Z8To. Mceric&Ad y y be he anton athigowld pller s s 


### The math trick for self attention

In [8]:

B, T, C = 4, 8, 2 # BATCH, TIME, CHANNEL
x = torch.randn(B, T, C)

x.shape

torch.Size([4, 8, 2])

In [9]:
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        prev = x[b, :t+1]
        xbow[b, t] = torch.mean(prev, 0)

In [10]:
print("X")
print(x[0])
print("xbow")
print(xbow[0])

X
tensor([[ 0.8958,  1.1688],
        [ 0.5941,  0.8015],
        [ 1.4922,  0.6182],
        [-0.6030,  0.5614],
        [-1.7053, -0.4459],
        [ 0.4562,  0.2048],
        [-0.1406,  0.4578],
        [ 0.1858,  1.7502]])
xbow
tensor([[0.8958, 1.1688],
        [0.7450, 0.9851],
        [0.9940, 0.8628],
        [0.5948, 0.7875],
        [0.1348, 0.5408],
        [0.1883, 0.4848],
        [0.1413, 0.4809],
        [0.1469, 0.6396]])


In [11]:
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()

print(a)
print(b)

a @ b


tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[7., 8.],
        [8., 2.],
        [5., 1.]])


tensor([[7.0000, 8.0000],
        [7.5000, 5.0000],
        [6.6667, 3.6667]])

This averages past tokens and prevents a token from seeing the future

In [22]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

### Attention mechanism

In [ ]:
B, T, C = 4, 8, 3 # BATCH, TIME, CHANNEL
x = torch.randn(B, T, C)

att_head_dim = 16

query = nn.Linear(C, att_head_dim, bias=False)
key = nn.Linear(C, att_head_dim, bias=False)
value = nn.Linear(C, att_head_dim, bias=False)


q = query(x) # B, T, att_head_dim
k = key(x) # B, T, att_head_dim
v = value(x) # B, T, att_head_dim

wei = (q @ torch.transpose(k, -2, -1)) # (B, T, att_head_dim)@(B, att_head_dim, T) = B, T, T

# After that we mask to prevent tokens from seeing the future

tril = torch.tril(torch.ones((T, T)))

wei = wei.masked_fill(tril == 0, float("-inf"))

wei = torch.softmax(wei * att_head_dim**-0.5, -1)


out = (wei @ v)

out.shape # B, T, att_head_dim

torch.Size([4, 8, 16])